# FD002 - Feature engineering condition-aware

Objetivo: analizar si habia una mejora distinta a seguir ajustando hiperparametros. La hipotesis viene de FD003: primero corregir por condicion operativa y despues agregar features temporales sensibles a degradacion.


## Lectura desde FD001 y FD003

FD001 tenia una sola condicion operativa y un solo modo de falla, por eso el LightGBM temporal con RUL cap 125 era una base fuerte. FD003 mostro que cambiar hiperparametros no movia mucho la aguja; la mejora vino de features fault-sensitive. FD002 combina seis condiciones operativas con trayectorias que cambian de regimen, asi que primero necesita normalizacion por condicion y recien despues features de degradacion.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[2] if Path.cwd().name == 'modeling' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
RESULTS_DIR = PROJECT_ROOT / 'results' / 'FD002'
NOTES_DIR = PROJECT_ROOT / 'notas' / 'hallazgos' / 'FD002'

ranking = pd.read_csv(RESULTS_DIR / 'fd002_final_candidate_ranking.csv')
feature_search = pd.read_csv(RESULTS_DIR / 'fd002_feature_engineering_search.csv')
ranking.head(10)[['experiment_stage','candidate_label','model_type','feature_set','window_size','sample_weight_scheme','cmapss_score','rmse','dangerous_error_pct']]


## Que se agrego

Se creo `feature_set='condition_fault_sensitive'`. Mantiene las features `condition_normalized` y suma pendientes, deltas, volatilidad, aceleracion e interacciones sobre sensores normalizados por condicion. Tambien se probo `mid_rul_guard`, un esquema de pesos para la banda 30-90 RUL, donde el XGB anterior tenia mas errores peligrosos.


In [ ]:
feature_search[['candidate_label','model_type','window_size','sample_weight_scheme','n_features','cmapss_score','rmse','dangerous_error_pct']]


In [ ]:
from src.fd001_modeling import metrics_by_rul_bin

old_preds = pd.read_csv(RESULTS_DIR / 'fd002_lgbm_xgb_model_comparison_predictions.csv')
old_preds = old_preds[old_preds['model_name'] == 'xgb_squarederror_condition_normalized_weighted']
new_preds = pd.read_csv(RESULTS_DIR / 'fd002_best_validation_predictions.csv')

old_bins = metrics_by_rul_bin(old_preds).assign(version='previous_xgb_condition_normalized')
new_bins = metrics_by_rul_bin(new_preds).assign(version='new_xgb_condition_fault_sensitive')
pd.concat([old_bins, new_bins], ignore_index=True)[['version','rul_bin','n_eval','mae','rmse','cmapss_score','dangerous_error_pct']]


## Decision

El candidato final interno queda como `xgb_condition_fault_sensitive_mid_guard`: C-MAPSS 982.893, RMSE 16.229 y dangerous error 9.69% en validacion artificial. Supera al XGB condition-normalized anterior en C-MAPSS y baja el riesgo de la banda media. El test oficial queda solo como reporte: baja dangerous error, pero sube C-MAPSS, por lo que el trade-off queda documentado en la nota escrita.


In [ ]:
print((NOTES_DIR / 'fd002_feature_engineering_interpretation.txt').read_text(encoding='utf-8'))
